# JAX backend

With the `jax` extra, `JaxModel` runs the rate evaluation, the population equations and the quasi-static reduction on JAX: jitted, vectorised with `vmap`, and differentiable. See [Backends](/OpenCR/guides/backends/) for what it covers and how it differs from the reference.

In [ ]:
import jax
import numpy as np
import plasma_plots

import opencr
from opencr.backends.jax_backend import JaxModel

model = opencr.load_model("hydrogen/atomic/basic", n_max=10)
conditions = opencr.Conditions(1.0, 1e18, densities={"H(n=1)": 1e19, "H+": 1e18})
jm = JaxModel(model, conditions)

## Jitted rates and a vmapped scan

The first call compiles; the second runs the compiled code:

In [ ]:
temperatures = np.geomspace(0.5, 100.0, 30)
densities = np.geomspace(1e16, 1e22, 5)
grid_t, grid_n = np.meshgrid(temperatures, densities, indexing="ij")
reduction = jm.reduction(grid_t, grid_n)  # compiles
reduction = jm.reduction(grid_t, grid_n)
s_cr = reduction.effective_rate("H(n=1)", "H+").reshape(grid_t.shape)
with plasma_plots.figure(1, 1) as fig:
    for j, density in enumerate(densities):
        fig[0].loglog(temperatures, s_cr[:, j], label=f"n_e = {density:.0e} m$^{{-3}}$")
    fig[0].set_xlabel("T_e (eV)")
    fig[0].set_ylabel(r"$S_\mathrm{CR}$ (m$^3$/s)")
    fig[0].set_title("effective ionization coefficient (JAX)")
    fig[0].legend(fontsize="small")

## Automatic differentiation

The Jacobian of the population equations by forward-mode AD agrees with the analytic sparse one, and `jax.grad` of an effective coefficient gives its sensitivity to the conditions, here $\partial \ln S_\mathrm{CR} / \partial \ln T_e$:

In [ ]:
system = model.system(conditions)
n = np.geomspace(1e10, 1e15, system.size)
analytic = system.jacobian(n).toarray()
print(
    f"Jacobian: largest difference {np.abs(np.asarray(jm.jacobian(n, 1.0, 1e18)) - analytic).max() / np.abs(analytic).max():.1e} of the largest entry"
)


def log_s_cr(log_t):
    return jax.numpy.log(jm.effective_rate("H(n=1)", "H+", jax.numpy.exp(log_t), 1e18))


slope = jax.jit(jax.grad(log_s_cr))
for t in (1.0, 3.0, 10.0, 30.0):
    print(f"T_e = {t:5.1f} eV: d ln S_CR / d ln T_e = {float(slope(np.log(t))):.2f}")